# Decision Trees

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Decision Tree Regression

In [2]:
# TODO: Write a regression tree class that can build tree and make prediction
class RegressionTree:
    def __init__(self, max_depth=3, min_samples=5):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.tree = None
        self.feature_names = None

    def _mse(self, y):
        if len(y) == 0:
            return 0.0
        return float(((y - np.mean(y)) ** 2).mean())

    def _find_best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_var = np.inf
        n = len(y)

        for feature in self.feature_names:
            x_feat = X[feature].values
            unique_vals = np.unique(x_feat)
            if len(unique_vals) < 2:
                continue
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

            for thresh in thresholds:
                left_mask = x_feat <= thresh
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]

                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                var_left = self._mse(y_left)
                var_right = self._mse(y_right)
                weighted_var = (len(y_left) / n) * var_left + (len(y_right) / n) * var_right

                if weighted_var < best_var:
                    best_var = weighted_var
                    best_threshold = thresh
                    best_feature = feature

        return best_feature, best_threshold, best_var

    def _build_tree(self, X, y, depth=0):
        # Stopping criteria: min samples reached or max depth reached
        if len(y) < self.min_samples or depth >= self.max_depth:
            return {'type': 'leaf', 'prediction': float(np.mean(y))}

        best_feature, best_threshold, _ = self._find_best_split(X, y)
        if best_feature is None:
            return {'type': 'leaf', 'prediction': float(np.mean(y))}

        left_mask = X[best_feature] <= best_threshold
        right_mask = ~left_mask

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return {
            'type': 'node',
            'feature': best_feature,
            'threshold': best_threshold,
            'left': left_child,
            'right': right_child
        }

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names = list(X.columns)
            X = X.copy()
        else:
            self.feature_names = [f'x{i}' for i in range(X.shape[1])]
            X = pd.DataFrame(X, columns=self.feature_names)

        if isinstance(y, (pd.Series, pd.DataFrame)):
            y = y.values.flatten()
        else:
            y = np.array(y).flatten()

        self.tree = self._build_tree(X, y)
        return self

    def _predict_sample(self, node, sample):
        if node['type'] == 'leaf':
            return node['prediction']

        feature = node['feature']
        threshold = node['threshold']

        if isinstance(sample, (dict, pd.Series)):
            val = sample[feature]
        else:
            idx = self.feature_names.index(feature)
            val = sample[idx]

        if val <= threshold:
            return self._predict_sample(node['left'], sample)
        else:
            return self._predict_sample(node['right'], sample)

    def predict(self, X):
        if isinstance(X, (dict, pd.Series)):
            return self._predict_sample(self.tree, X)
        if isinstance(X, pd.DataFrame):
            return np.array([self._predict_sample(self.tree, row) for _, row in X.iterrows()])
        X_arr = np.array(X)
        if X_arr.ndim == 1:
            return self._predict_sample(self.tree, X_arr)
        return np.array([self._predict_sample(self.tree, row) for row in X_arr])

In [3]:
df = pd.read_csv('../data/Lab4/study_score.csv')
df

,StudyHours,SleepHours,CoffeeCups,Score,Pass
0,1,9,0,3,0
1,2,8,1,5,0
2,3,7,1,8,1
3,4,7,2,12,1
4,5,6,2,15,1
5,6,6,3,14,1
6,7,5,3,10,1
7,8,5,4,7,0
8,9,4,4,4,0
9,10,4,5,2,0


In [4]:
# Train and evaluate RegressionTree
features = ['StudyHours', 'SleepHours', 'CoffeeCups']
X_reg = df[features]
y_reg = df['Score']

reg_tree = RegressionTree(max_depth=3, min_samples=5)
reg_tree.fit(X_reg, y_reg)

# Predict for a new student: 10 study hours, 6 sleep hours, 3 coffee cups
sample = {'StudyHours': 10, 'SleepHours': 6, 'CoffeeCups': 3}
pred_sample = reg_tree.predict(sample)
print(f"Prediction for sample {sample}: {pred_sample:.2f}")

# Predict on the entire dataset
y_reg_pred = reg_tree.predict(X_reg)
mse_val = np.mean((y_reg - y_reg_pred) ** 2)

results_reg = pd.DataFrame({
    'Actual Score': y_reg,
    'Predicted Score': y_reg_pred
})
print(f"\nMean Squared Error (MSE): {mse_val:.3f}")
print("\nActual vs Predicted Scores:")
print(results_reg)

Prediction for sample {'StudyHours': 10, 'SleepHours': 6, 'CoffeeCups': 3}: 3.00

Mean Squared Error (MSE): 2.383

Actual vs Predicted Scores:
   Actual Score  Predicted Score
0             3         5.333333
1             5         5.333333
2             8         5.333333
3            12        13.666667
4            15        13.666667
5            14        13.666667
6            10         8.500000
7             7         8.500000
8             4         3.000000
9             2         3.000000


## 2. Decision Classification

In [5]:
# TODO: Write a classificatin tree class that can build tree and make prediction
class ClassificationTree:
    def __init__(self, max_depth=3, min_samples=5):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.tree = None
        self.feature_names = None

    def _gini_impurity(self, y):
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        proportions = counts / len(y)
        return float(1.0 - np.sum(proportions ** 2))

    def _majority_class(self, y):
        vals, counts = np.unique(y, return_counts=True)
        return vals[np.argmax(counts)]

    def _find_best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_gini = np.inf
        n = len(y)

        for feature in self.feature_names:
            x_feat = X[feature].values
            unique_vals = np.unique(x_feat)
            if len(unique_vals) < 2:
                continue
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

            for thresh in thresholds:
                left_mask = x_feat <= thresh
                right_mask = ~left_mask

                y_left = y[left_mask]
                y_right = y[right_mask]

                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                gini_left = self._gini_impurity(y_left)
                gini_right = self._gini_impurity(y_right)
                weighted_gini = (len(y_left) / n) * gini_left + (len(y_right) / n) * gini_right

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_threshold = thresh
                    best_feature = feature

        return best_feature, best_threshold, best_gini

    def _build_tree(self, X, y, depth=0):
        # Stopping criteria: pure node, min samples reached, or max depth reached
        if len(np.unique(y)) == 1:
            return {'type': 'leaf', 'prediction': y[0]}
        if len(y) < self.min_samples or depth >= self.max_depth:
            return {'type': 'leaf', 'prediction': self._majority_class(y)}

        best_feature, best_threshold, _ = self._find_best_split(X, y)
        if best_feature is None:
            return {'type': 'leaf', 'prediction': self._majority_class(y)}

        left_mask = X[best_feature] <= best_threshold
        right_mask = ~left_mask

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return {
            'type': 'node',
            'feature': best_feature,
            'threshold': best_threshold,
            'left': left_child,
            'right': right_child
        }

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names = list(X.columns)
            X = X.copy()
        else:
            self.feature_names = [f'x{i}' for i in range(X.shape[1])]
            X = pd.DataFrame(X, columns=self.feature_names)

        if isinstance(y, (pd.Series, pd.DataFrame)):
            y = y.values.flatten()
        else:
            y = np.array(y).flatten()

        self.tree = self._build_tree(X, y)
        return self

    def _predict_sample(self, node, sample):
        if node['type'] == 'leaf':
            return node['prediction']

        feature = node['feature']
        threshold = node['threshold']

        if isinstance(sample, (dict, pd.Series)):
            val = sample[feature]
        else:
            idx = self.feature_names.index(feature)
            val = sample[idx]

        if val <= threshold:
            return self._predict_sample(node['left'], sample)
        else:
            return self._predict_sample(node['right'], sample)

    def predict(self, X):
        if isinstance(X, (dict, pd.Series)):
            return self._predict_sample(self.tree, X)
        if isinstance(X, pd.DataFrame):
            return np.array([self._predict_sample(self.tree, row) for _, row in X.iterrows()])
        X_arr = np.array(X)
        if X_arr.ndim == 1:
            return self._predict_sample(self.tree, X_arr)
        return np.array([self._predict_sample(self.tree, row) for row in X_arr])

In [ ]:
# Train and evaluate ClassificationTree
y_cls = df['Pass']

cls_tree = ClassificationTree(max_depth=3, min_samples=5)
cls_tree.fit(X_reg, y_cls)

# Predict for the sample
pred_cls_sample = cls_tree.predict(sample)
outcome = 'Pass (1)' if pred_cls_sample == 1 else 'Fail (0)'
print(f"Prediction for sample {sample}: {outcome}")

# Predict on the entire dataset
y_cls_pred = cls_tree.predict(X_reg)
accuracy = np.mean(y_cls == y_cls_pred)

results_cls = pd.DataFrame({
    'Actual Pass': y_cls,
    'Predicted Pass': y_cls_pred
})
print(f"\nClassification Accuracy: {accuracy * 100:.1f}%")
print("\nActual vs Predicted Pass:")
print(results_cls)

Prediction for sample {'StudyHours': 10, 'SleepHours': 6, 'CoffeeCups': 3}: Fail (0)

Classification Accuracy: 100.0%

Actual vs Predicted Pass:
   Actual Pass  Predicted Pass
0            0               0
1            0               0
2            1               1
3            1               1
4            1               1
5            1               1
6            1               1
7            0               0
8            0               0
9            0               0


: 